In [3]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/competitions/playground-series-s6e3/sample_submission.csv
/kaggle/input/competitions/playground-series-s6e3/train.csv
/kaggle/input/competitions/playground-series-s6e3/test.csv
/kaggle/input/datasets/blastchar/telco-customer-churn/WA_Fn-UseC_-Telco-Customer-Churn.csv


In [4]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score

# ── Load Competition Data ──────────────────────────────
train = pd.read_csv('/kaggle/input/competitions/playground-series-s6e3/train.csv')
test  = pd.read_csv('/kaggle/input/competitions/playground-series-s6e3/test.csv')
sub   = pd.read_csv('/kaggle/input/competitions/playground-series-s6e3/sample_submission.csv')

# ── Load Original Dataset (extra training data!) ───────
try:
    orig = pd.read_csv('/kaggle/input/datasets/blastchar/telco-customer-churn/WA_Fn-UseC_-Telco-Customer-Churn.csv')
    orig = orig.drop(columns=['customerID'], errors='ignore')
    orig['TotalCharges'] = pd.to_numeric(orig['TotalCharges'], errors='coerce')
    orig['Churn']        = orig['Churn'].map({'Yes': 1, 'No': 0})
    orig['id']           = -1  # mark as original
    has_orig             = True
    print(f'✅ Original dataset loaded: {orig.shape}')
except:
    has_orig = False
    print('⚠️ Original dataset not found — add telco-customer-churn to notebook inputs')

# ── Fix Target ─────────────────────────────────────────
if train['Churn'].dtype == object:
    train['Churn'] = train['Churn'].map({'Yes': 1, 'No': 0})
train['Churn'] = train['Churn'].astype(int)

# ── Merge original into train ──────────────────────────
if has_orig:
    # Align columns
    orig_cols = [c for c in train.columns if c in orig.columns]
    orig_aligned = orig[orig_cols].copy()
    train = pd.concat([train, orig_aligned], axis=0, ignore_index=True)
    print(f'✅ Train after adding original: {train.shape}')

print(f'Train : {train.shape} | Test : {test.shape}')
print(f'Churn balance:\n{train["Churn"].value_counts(normalize=True).round(3)}')

TARGET   = 'Churn'
ID_COL   = 'id'
N_SPLITS = 10
SEED     = 42
skf      = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)

# ── Feature Engineering ────────────────────────────────
def feature_engineering(df):
    df = df.copy()

    # Force convert all to numeric or encode
    cat_cols = df.select_dtypes(include='object').columns.tolist()
    cat_cols = [c for c in cat_cols if c not in [ID_COL, TARGET]]
    for col in cat_cols:
        df[col] = LabelEncoder().fit_transform(df[col].astype(str))

    # Force numeric
    for col in df.columns:
        if col in [ID_COL, TARGET]: continue
        df[col] = pd.to_numeric(df[col], errors='coerce')

    # Fill NaN
    df = df.fillna(df.median(numeric_only=True))

    # Interactions
    if 'tenure' in df.columns and 'MonthlyCharges' in df.columns:
        df['charge_per_tenure'] = df['MonthlyCharges'] / (df['tenure'] + 1)
        df['total_charge_est']  = df['MonthlyCharges'] * df['tenure']
        df['tenure_x_monthly']  = df['tenure'] * df['MonthlyCharges']

    if 'TotalCharges' in df.columns and 'MonthlyCharges' in df.columns:
        df['charge_ratio']  = df['TotalCharges'] / (df['MonthlyCharges'] + 1)
        df['charges_diff']  = df['TotalCharges'] - df['MonthlyCharges'] * df['tenure']

    if 'tenure' in df.columns:
        df['tenure_squared']   = df['tenure'] ** 2
        df['tenure_log']       = np.log1p(df['tenure'])
        df['is_new_customer']  = (df['tenure'] <= 3).astype(int)
        df['is_long_customer'] = (df['tenure'] >= 48).astype(int)

    if 'MonthlyCharges' in df.columns:
        df['monthly_log']   = np.log1p(df['MonthlyCharges'])
        df['is_high_payer'] = (df['MonthlyCharges'] > 65).astype(int)

    if 'TotalCharges' in df.columns:
        df['total_log'] = np.log1p(df['TotalCharges'])

    # Service count
    service_cols = [c for c in df.columns if c in [
        'PhoneService','MultipleLines','InternetService',
        'OnlineSecurity','OnlineBackup','DeviceProtection',
        'TechSupport','StreamingTV','StreamingMovies'
    ]]
    if service_cols:
        df['num_services'] = df[service_cols].sum(axis=1)

    # Aggregate stats
    num_cols = df.select_dtypes(include=np.number).columns.tolist()
    num_cols = [c for c in num_cols if c not in [ID_COL, TARGET]]
    df['num_mean'] = df[num_cols].mean(axis=1)
    df['num_std']  = df[num_cols].std(axis=1)
    df['num_max']  = df[num_cols].max(axis=1)
    df['num_min']  = df[num_cols].min(axis=1)

    return df

train_fe = feature_engineering(train)
test_fe  = feature_engineering(test)

FEATURES = [c for c in train_fe.columns if c not in [ID_COL, TARGET]]
X        = train_fe[FEATURES]
y        = train_fe[TARGET]
X_test   = test_fe[FEATURES]

# Align columns
X_test = X_test.reindex(columns=FEATURES, fill_value=0)

print(f'\n✅ Total features : {len(FEATURES)}')
print(f'X      : {X.shape}')
print(f'X_test : {X_test.shape}')

✅ Original dataset loaded: (7043, 21)
✅ Train after adding original: (601237, 21)
Train : (601237, 21) | Test : (254655, 20)
Churn balance:
Churn
0    0.774
1    0.226
Name: proportion, dtype: float64

✅ Total features : 36
X      : (601237, 36)
X_test : (254655, 36)


In [5]:
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout)

Sat Mar 21 18:27:39 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.105.08             Driver Version: 580.105.08     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla P100-PCIE-16GB           Off |   00000000:00:04.0 Off |                    0 |
| N/A   36C    P0             26W /  250W |       0MiB /  16384MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [6]:
from xgboost  import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
import lightgbm as lgb

oof_xgb  = np.zeros(len(X))
oof_lgb  = np.zeros(len(X))
oof_cat  = np.zeros(len(X))
pred_xgb = np.zeros(len(X_test))
pred_lgb = np.zeros(len(X_test))
pred_cat = np.zeros(len(X_test))

# ── XGBoost ───────────────────────────────────────────
print('='*50)
print('🚀 Training XGBoost...')
print('='*50)

xgb_params = dict(
    n_estimators          = 2000,
    learning_rate         = 0.03,
    max_depth             = 6,
    min_child_weight      = 5,
    gamma                 = 0.1,
    subsample             = 0.8,
    colsample_bytree      = 0.8,
    reg_alpha             = 0.1,
    reg_lambda            = 1.0,
    scale_pos_weight      = (y==0).sum() / (y==1).sum(),
    tree_method           = 'hist',
    device                = 'cuda',
    eval_metric           = 'auc',
    early_stopping_rounds = 100,
    random_state          = SEED,
    n_jobs                = -1
)

for fold, (tr_idx, val_idx) in enumerate(skf.split(X, y)):
    X_tr, X_val = X.iloc[tr_idx], X.iloc[val_idx]
    y_tr, y_val = y.iloc[tr_idx], y.iloc[val_idx]

    model = XGBClassifier(**xgb_params)
    model.fit(X_tr, y_tr,
              eval_set=[(X_val, y_val)],
              verbose=False)

    oof_xgb[val_idx]  = model.predict_proba(X_val)[:, 1]
    pred_xgb         += model.predict_proba(X_test)[:, 1] / N_SPLITS
    print(f'  Fold {fold+1:2d} | AUC: {roc_auc_score(y_val, oof_xgb[val_idx]):.5f}')

print(f'\n📊 XGBoost OOF AUC: {roc_auc_score(y, oof_xgb):.5f}')

# ── LightGBM ──────────────────────────────────────────
print('\n' + '='*50)
print('🚀 Training LightGBM...')
print('='*50)

lgb_params = dict(
    n_estimators      = 2000,
    learning_rate     = 0.03,
    num_leaves        = 63,
    max_depth         = -1,
    min_child_samples = 20,
    feature_fraction  = 0.8,
    bagging_fraction  = 0.8,
    bagging_freq      = 5,
    reg_alpha         = 0.1,
    reg_lambda        = 1.0,
    is_unbalance      = True,
    random_state      = SEED,
    n_jobs            = -1,
    verbosity         = -1
)

for fold, (tr_idx, val_idx) in enumerate(skf.split(X, y)):
    X_tr, X_val = X.iloc[tr_idx], X.iloc[val_idx]
    y_tr, y_val = y.iloc[tr_idx], y.iloc[val_idx]

    model = LGBMClassifier(**lgb_params)
    model.fit(X_tr, y_tr,
              eval_set    = [(X_val, y_val)],
              eval_metric = 'auc',
              callbacks   = [
                  lgb.early_stopping(100, verbose=False),
                  lgb.log_evaluation(False)
              ])

    oof_lgb[val_idx]  = model.predict_proba(X_val)[:, 1]
    pred_lgb         += model.predict_proba(X_test)[:, 1] / N_SPLITS
    print(f'  Fold {fold+1:2d} | AUC: {roc_auc_score(y_val, oof_lgb[val_idx]):.5f}')

print(f'\n📊 LightGBM OOF AUC: {roc_auc_score(y, oof_lgb):.5f}')

# ── CatBoost ──────────────────────────────────────────
print('\n' + '='*50)
print('🚀 Training CatBoost...')
print('='*50)

cat_params = dict(
    iterations         = 2000,
    learning_rate      = 0.03,
    depth              = 6,
    l2_leaf_reg        = 3,
    bootstrap_type     = 'Bernoulli',
    subsample          = 0.8,
    min_data_in_leaf   = 20,
    auto_class_weights = 'Balanced',
    eval_metric        = 'AUC',
    task_type          = 'GPU',
    random_seed        = SEED,
    verbose            = False
)

for fold, (tr_idx, val_idx) in enumerate(skf.split(X, y)):
    X_tr, X_val = X.iloc[tr_idx], X.iloc[val_idx]
    y_tr, y_val = y.iloc[tr_idx], y.iloc[val_idx]

    model = CatBoostClassifier(**cat_params)
    model.fit(X_tr, y_tr,
              eval_set              = (X_val, y_val),
              early_stopping_rounds = 100)

    oof_cat[val_idx]  = model.predict_proba(X_val)[:, 1]
    pred_cat         += model.predict_proba(X_test)[:, 1] / N_SPLITS
    print(f'  Fold {fold+1:2d} | AUC: {roc_auc_score(y_val, oof_cat[val_idx]):.5f}')

print(f'\n📊 CatBoost OOF AUC: {roc_auc_score(y, oof_cat):.5f}')

# ── Summary ───────────────────────────────────────────
print('\n' + '='*50)
print('📊 GBDT Summary')
print('='*50)
print(f'  XGBoost  OOF AUC : {roc_auc_score(y, oof_xgb):.5f}')
print(f'  LightGBM OOF AUC : {roc_auc_score(y, oof_lgb):.5f}')
print(f'  CatBoost OOF AUC : {roc_auc_score(y, oof_cat):.5f}')

🚀 Training XGBoost...
  Fold  1 | AUC: 0.91533
  Fold  2 | AUC: 0.91628
  Fold  3 | AUC: 0.91832
  Fold  4 | AUC: 0.91439
  Fold  5 | AUC: 0.91471
  Fold  6 | AUC: 0.91359
  Fold  7 | AUC: 0.91545
  Fold  8 | AUC: 0.91508
  Fold  9 | AUC: 0.91624
  Fold 10 | AUC: 0.91623

📊 XGBoost OOF AUC: 0.91556

🚀 Training LightGBM...
  Fold  1 | AUC: 0.91164
  Fold  2 | AUC: 0.91238
  Fold  3 | AUC: 0.91410
  Fold  4 | AUC: 0.90976
  Fold  5 | AUC: 0.91055
  Fold  6 | AUC: 0.90971
  Fold  7 | AUC: 0.91135
  Fold  8 | AUC: 0.91088
  Fold  9 | AUC: 0.91222
  Fold 10 | AUC: 0.91120

📊 LightGBM OOF AUC: 0.91129

🚀 Training CatBoost...


Default metric period is 5 because AUC is/are not implemented for GPU


  Fold  1 | AUC: 0.91534


Default metric period is 5 because AUC is/are not implemented for GPU


  Fold  2 | AUC: 0.91600


...

  Fold  8 | AUC: 0.91498


Default metric period is 5 because AUC is/are not implemented for GPU


  Fold  9 | AUC: 0.91584


Default metric period is 5 because AUC is/are not implemented for GPU


  Fold 10 | AUC: 0.91589

📊 CatBoost OOF AUC: 0.91534

📊 GBDT Summary
  XGBoost  OOF AUC : 0.91556
  LightGBM OOF AUC : 0.91129
  CatBoost OOF AUC : 0.91534


In [7]:
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)
from sklearn.linear_model import LogisticRegression

# ── Stack OOFs ────────────────────────────────────────
names       = ['XGBoost', 'LightGBM', 'CatBoost']
oof_matrix  = np.column_stack([oof_xgb, oof_lgb, oof_cat])
pred_matrix = np.column_stack([pred_xgb, pred_lgb, pred_cat])

# ── Optuna Weight Search ──────────────────────────────
print('🔍 Running Optuna weight search (1000 trials)...')

def objective(trial):
    w = np.array([trial.suggest_float(f'w{i}', 0.0, 1.0) for i in range(3)])
    w = w / w.sum()
    return roc_auc_score(y, (oof_matrix * w).sum(axis=1))

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=1000, show_progress_bar=True)

best_w = np.array([study.best_params[f'w{i}'] for i in range(3)])
best_w = best_w / best_w.sum()

print(f'\n⚖️  Optimal weights:')
for n, w in zip(names, best_w):
    print(f'  {n:12s}: {w:.4f}')

oof_blend  = (oof_matrix  * best_w).sum(axis=1)
pred_blend = (pred_matrix * best_w).sum(axis=1)
print(f'\n📊 Weighted Blend OOF AUC : {roc_auc_score(y, oof_blend):.5f}')

# ── Level-2 Stacker ───────────────────────────────────
print('\n🚀 Training Level-2 Stacker...')

oof_stack  = np.zeros(len(y))
pred_stack = np.zeros(len(X_test))

for fold, (tr_idx, val_idx) in enumerate(skf.split(oof_matrix, y)):
    meta = LogisticRegression(C=1.0, max_iter=1000)
    meta.fit(oof_matrix[tr_idx], y.iloc[tr_idx])
    oof_stack[val_idx]  = meta.predict_proba(oof_matrix[val_idx])[:, 1]
    pred_stack         += meta.predict_proba(pred_matrix)[:, 1] / N_SPLITS

print(f'📊 Level-2 Stack OOF AUC  : {roc_auc_score(y, oof_stack):.5f}')

# ── Final Blend ───────────────────────────────────────
FINAL_W    = 0.6
oof_final  = FINAL_W * oof_blend  + (1 - FINAL_W) * oof_stack
pred_final = FINAL_W * pred_blend + (1 - FINAL_W) * pred_stack

print(f'\n🏆 FINAL OOF AUC : {roc_auc_score(y, oof_final):.5f}')

# ── Submit ────────────────────────────────────────────
sub['Churn'] = pred_final
sub.to_csv('submission.csv', index=False)

print(f'\n✅ submission.csv saved — {len(sub)} rows')
print(f'  Mean : {pred_final.mean():.4f}')
print(f'  Std  : {pred_final.std():.4f}')
print(f'  Min  : {pred_final.min():.4f}')
print(f'  Max  : {pred_final.max():.4f}')

# ── Final Summary ─────────────────────────────────────
print('\n' + '='*50)
print('🏆 FINAL SUMMARY')
print('='*50)
print(f'  XGBoost  OOF AUC : {roc_auc_score(y, oof_xgb):.5f}')
print(f'  LightGBM OOF AUC : {roc_auc_score(y, oof_lgb):.5f}')
print(f'  CatBoost OOF AUC : {roc_auc_score(y, oof_cat):.5f}')
print(f'  Blend    OOF AUC : {roc_auc_score(y, oof_blend):.5f}')
print(f'  Stack    OOF AUC : {roc_auc_score(y, oof_stack):.5f}')
print(f'  ─────────────────────────────')
print(f'  FINAL    OOF AUC : {roc_auc_score(y, oof_final):.5f} 🎯')
print('='*50)
print('\n✅ DONE — Output tab → submission.csv → Submit!')

🔍 Running Optuna weight search (1000 trials)...


  0%|          | 0/1000 [00:00<?, ?it/s]


⚖️  Optimal weights:
  XGBoost     : 0.6309
  LightGBM    : 0.0000
  CatBoost    : 0.3691

📊 Weighted Blend OOF AUC : 0.91566

🚀 Training Level-2 Stacker...
📊 Level-2 Stack OOF AUC  : 0.91565

🏆 FINAL OOF AUC : 0.91566

✅ submission.csv saved — 254655 rows
  Mean : 0.2926
  Std  : 0.3156
  Min  : 0.0055
  Max  : 0.9287

🏆 FINAL SUMMARY
  XGBoost  OOF AUC : 0.91556
  LightGBM OOF AUC : 0.91129
  CatBoost OOF AUC : 0.91534
  Blend    OOF AUC : 0.91566
  Stack    OOF AUC : 0.91565
  ─────────────────────────────
  FINAL    OOF AUC : 0.91566 🎯

✅ DONE — Output tab → submission.csv → Submit!
